<div align="center">
  <h3><b>ESCUELA POLITÉCNICA NACIONAL</b></h3>
  <h3><b>FACULTAD DE INGENIERÍA EN SISTEMAS</b></h3>
  <h3><b>INGENIERÍA EN CIENCIAS DE LA COMPUTACIÓN</b></h3>
  <h3><b>RECUPERACIÓN DE LA INFORMACIÓN</b></h3>
</div>

---
**Nombre**   Mark Hernández        
**Fecha**    13/05/26  
**Docente**  Iván Carrera

# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


Para cargar el corpus primero tenemos que cargar el archivo csv.

### Carga del archivo csv


In [18]:
import pandas as pd
import kagglehub
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

# Descargar recursos de NLTK
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# 1. Carga del corpus (Wikipedia Text Corpus)
path = kagglehub.dataset_download("gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects")
# Asumiendo que el dataset contiene un CSV o archivos de texto, cargamos una muestra
df = pd.read_csv(f"{path}/wikipedia_text_corpus.csv", index_col=0).head(1000)



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mark_\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mark_\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mark_\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Luego de cargar el corpus nos dedicamos al preprocesamiento.

In [19]:
import re

# Inicialización de herramientas
stop_words = set(stopwords.words('english'))
stemmer = SnowballStemmer('english')

def preprocess(text):
    if not isinstance(text, str):
        return ""
    # Limpieza básica
    text = re.sub(r"[^a-zA-Z0-9\s]", '', text.lower())
    # Tokenización y filtrado
    tokens = word_tokenize(text)
    filtered = [stemmer.stem(w) for w in tokens if w not in stop_words]
    return " ".join(filtered)

# Aplicar al dataframe
df['clean_text'] = df['text'].apply(preprocess)

Mostramos los 5 primeros documentos

In [20]:
df.head(10)

,text,clean_text
1,Anovo\n\nAnovo (formerly A Novo) is a computer...,anovo anovo former novo comput servic compani ...
2,Battery indicator\n\nA battery indicator (also...,batteri indic batteri indic also known batteri...
3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...",bob peas robert allen peas august 22 1940 june...
4,CAVNET\n\nCAVNET was a secure military forum w...,cavnet cavnet secur militari forum becam oper ...
5,CLidar\n\nThe CLidar is a scientific instrumen...,clidar clidar scientif instrument use measur p...
6,Capacity loss\n\nCapacity loss or capacity fad...,capac loss capac loss capac fade phenomenon ob...
7,Carbon Recycling International\n\nCarbon Recyc...,carbon recycl intern carbon recycl intern cri ...
8,Chemical Agent Resistant Coating\n\nChemical A...,chemic agent resist coat chemic agent resist c...
9,Claas Cougar\n\nThe Claas Cougar is a self-pro...,claa cougar claa cougar selfpropel mower produ...
10,"Conductive polymer\n\nConductive polymers or, ...",conduct polym conduct polym precis intrins con...


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 3. Representación Vectorial
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['clean_text'])



,00,000,000015,00001g,0001,00025t,000431,00044,00061,00074,...,zurich,zurovec,zuse,zx,zx81,zygosaccharomyc,zymo,zynga,zyyx,zz
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# 4. Verificación con 10 queries
queries = ["artificial intelligence history", "computer science evolution", "space exploration", ...] # Define tus 10 queries

def search_tfidf(query, top_n=5):
    query_clean = preprocess(query)
    query_vec = vectorizer.transform([query_clean])
    sim = cosine_similarity(query_vec, tfidf_matrix).flatten()
    indices = sim.argsort()[-top_n:][::-1]
    return df.iloc[indices]

## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 